In [1]:
import pandas as pd
from pathlib import Path

# --- 1. CONFIGURACIÓN ---
CSV_FOLDER = Path(r"C:\Users\david\Desktop\Results\datos_csv") 
all_files = sorted(list(CSV_FOLDER.glob('*.csv')))

def auditar_dataset(files):
    print(f"🕵️‍♂️ Iniciando auditoría de {len(files)} archivos...\n")
    
    errores = []
    total_frames = 0
    archivos_perfectos = 0
    
    for file_path in files:
        try:
            df = pd.read_csv(file_path)
            
            # Chequeo 1: ¿Está vacío?
            if df.empty:
                errores.append(f"❌ {file_path.name}: Archivo VACÍO.")
                continue
                
            # Chequeo 2: ¿Tiene NaNs?
            nulos = df.isnull().sum().sum()
            if nulos > 0:
                errores.append(f"⚠️ {file_path.name}: Tiene {nulos} valores NaN (Huecos sin rellenar).")
            
            # Chequeo 3: ¿Es demasiado corto? (Menos de 30 frames = 1 segundo)
            if len(df) < 30:
                errores.append(f"⚠️ {file_path.name}: Muy corto ({len(df)} frames).")
                
            # Chequeo 4: Integridad de coordenadas (0.0 a 1.0)
            # MediaPipe normalizado debe estar entre 0 y 1 (aprox, a veces se sale un poco)
            # Si hay un valor > 2.0 o < -1.0 es sospechoso
            cols_coords = [c for c in df.columns if c.endswith('_x') or c.endswith('_y')]
            if not df[cols_coords].empty:
                max_val = df[cols_coords].max().max()
                min_val = df[cols_coords].min().min()
                
                if max_val > 2.0 or min_val < -1.0:
                    errores.append(f"❓ {file_path.name}: Coordenadas raras (Max: {max_val:.2f}, Min: {min_val:.2f})")
            
            # Si pasa todo
            if nulos == 0 and len(df) >= 30:
                archivos_perfectos += 1
                total_frames += len(df)
                
        except Exception as e:
            errores.append(f"☠️ {file_path.name}: Error al leer ({e})")

    print("-" * 50)
    print(f"✅ Auditoría Finalizada.")
    print(f"📊 Archivos Perfectos: {archivos_perfectos} / {len(files)}")
    print(f"🎞️ Total Frames Procesados: {total_frames}")
    
    if errores:
        print(f"\n🚨 SE HAN ENCONTRADO {len(errores)} INCIDENCIAS:")
        for err in errores:
            print(err)
    else:
        print("\n✨ ¡ENHORABUENA! Tu dataset está impoluto. 0 Errores.")

# --- EJECUTAR AUDITORÍA ---
auditar_dataset(all_files)

🕵️‍♂️ Iniciando auditoría de 109 archivos...

--------------------------------------------------
✅ Auditoría Finalizada.
📊 Archivos Perfectos: 109 / 109
🎞️ Total Frames Procesados: 44532

✨ ¡ENHORABUENA! Tu dataset está impoluto. 0 Errores.
